In [0]:
%sql

SELECT COUNT(*) FROM
bootcamp.silver.propiedades;
DESCRIBE TABLE bootcamp.silver.propiedades;

In [0]:
%sql
use bootcamp.silver;

select count(*),
 count(distinct (estado)) as estado,
count(distinct (partido)) as partido,
count(distinct (region)) as region,
count(distinct (tipo_operacion)) as tipo_operacion,
count(distinct (precio)) as precio,
count(distinct (moneda)) as moneda,
count(distinct (ambientes)) as ambientes,
count(distinct (cochera)) as cochera,
count(distinct (orientacion)) as orientacion,
count(distinct (url)) as url

 from bootcamp.silver.propiedades

In [0]:
%sql

CREATE or REPLACE TEMP VIEW stagin_correcciones AS
    SELECT partido,region,'Gran Buenos Aires' AS ciudad
    from bootcamp.gold.dim_zona
    WHERE region ='gba zona norte' and ciudad = 'GBA';

select * from stagin_correcciones


In [0]:
%sql

SELECT COUNT(*) AS count_antes FROM bootcamp.gold.dim_zona;


In [0]:
%sql

merge into bootcamp.gold.dim_zona AS Target
USING stagin_correcciones as SOURCE
ON target.partido=source.partido and target.region=source.region
when matched then
UPDATE SET ciudad = source.ciudad; 

In [0]:
%sql

drop table bootcamp.gold.dim_zona_scd2

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd2  (
zona_id BIGINT GENERATED ALWAYS AS IDENTITY,
partido string not NULL,
region STRING not null,
ciudad STRING,
provincia string DEFAULT "Buenos Aires",
pais STRING default "Argentina",
valid_from TIMESTAMP not null default current_timestamp(),
valid_to TIMESTAMP default "9999-12-31",
is_current boolean default true
) 

USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')
;



In [0]:
%sql

INSERT INTO bootcamp.gold.dim_zona_scd2 (
    partido,
    region,
    ciudad
)
SELECT distinct partido, region,
case when region = 'capital federal' then 'CABA'
else 'GBA' end as ciudad
from bootcamp.silver.propiedades
WHERE partido IS NOT NULL;

In [0]:
%sql

SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN is_current = TRUE THEN 1 ELSE 0 END) AS actuales,
  SUM(CASE WHEN is_current = FALSE THEN 1 ELSE 0 END) AS historicos
FROM bootcamp.gold.dim_zona_scd2;

In [0]:
%sql

SELECT * from bootcamp.gold.dim_zona_scd2

In [0]:
%sql

update  bootcamp.gold.dim_zona_scd2
set valid_to = current_timestamp(), is_current=false
where partido = 'capital federal' and ciudad = 'CABA' and is_current=true


In [0]:
%sql
-- Paso 3: Verificar que todos son is_current = TRUE
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN is_current = TRUE THEN 1 ELSE 0 END) AS actuales,
  SUM(CASE WHEN is_current = FALSE THEN 1 ELSE 0 END) AS historicos
FROM bootcamp.gold.dim_zona_scd2;

In [0]:
%sql

update  bootcamp.gold.dim_zona_scd2
set valid_to = current_timestamp(), is_current=false
where partido = 'capital federal' and ciudad = 'CABA' and is_current=true

In [0]:
%sql
insert into bootcamp.gold.dim_zona_scd2 (
    partido,region,ciudad,
    valid_from,valid_to,is_current
)
values (
    'capital federal','capital federal','Buenos Aires',
    current_timestamp(),'9999-12-31',TRUE
)

In [0]:
%sql

SELECT * FROM
bootcamp.gold.dim_zona_scd2 WHERE partido = 'capital federal'
ORDER BY valid_from

In [0]:
%sql

create or replace temp view staging_cambios_scd2  as
select *  from VALUES
('palermo', 'capital federal', 'CABA Sur'),
  ('barrio-nuevo', 'capital federal', 'CABA')
AS t(partido, region, ciudad_nueva)


In [0]:
%sql

MERGE into bootcamp.gold.dim_zona_scd2 TARGET
USING staging_cambios_scd2 SOURCE
ON target.partido = source.partido
   AND target.region = source.region
   AND target.is_current = TRUE
WHEN MATCHED AND target.ciudad != source.ciudad_nueva THEN
  UPDATE SET valid_to = CURRENT_TIMESTAMP(), is_current = FALSE
WHEN NOT MATCHED THEN
  INSERT (partido, region, ciudad, valid_from, valid_to, is_current)
  VALUES (source.partido, source.region, source.ciudad_nueva, CURRENT_TIMESTAMP(), '9999-12-31', TRUE);

In [0]:
%sql
-- Paso 3: INSERT nuevas versiones para los que se cerraron
INSERT INTO bootcamp.gold.dim_zona_scd2 (partido, region, ciudad, valid_from, valid_to, is_current)
SELECT s.partido, s.region, s.ciudad_nueva, CURRENT_TIMESTAMP(), '9999-12-31', TRUE
FROM staging_cambios_scd2 s
INNER JOIN bootcamp.gold.dim_zona_scd2 t
  ON s.partido = t.partido AND s.region = t.region
  AND t.is_current = FALSE AND t.valid_to >= CURRENT_TIMESTAMP() - INTERVAL 1 MINUTE
WHERE NOT EXISTS (
  SELECT 1 FROM bootcamp.gold.dim_zona_scd2 t2
  WHERE t2.partido = s.partido AND t2.region = s.region AND t2.is_current = TRUE
);

In [0]:
%sql
-- Paso 4: Verificar - palermo debe tener 2 versiones, barrio-nuevo 1
SELECT partido, region, ciudad, valid_from, valid_to, is_current
FROM bootcamp.gold.dim_zona_scd2
WHERE partido IN ('palermo', 'barrio-nuevo')
ORDER BY partido, valid_from;

In [0]:
%sql

CREATE TABLE IF NOT EXISTs bootcamp.gold.dim_zona_scd3 (

    zona_id bigint GENERATED ALWAYS AS IDENTITY,
    partido string not null,
    region string not null,
    ciudad string,
    ciudad_anterior string,
    fecha_ultima_actualizacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    provincia        STRING DEFAULT 'Buenos Aires',
  pais             STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (zona_id)
   
) USING DELTA
COMMENT 'Dimension de zonas -- SCD Type 3 (valor actual + anterior)'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql

INSERT INTO bootcamp.gold.dim_zona_scd3 (
    partido,region,ciudad)
    SELECT distinct partido, region,
    CASE WHEN region = "capital federal" then "CABA"
    ELSE "GBA" end
from bootcamp.silver.propiedades
where partido is not null

In [0]:
%sql
select * from bootcamp.gold.dim_zona_scd3

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW source_cambios AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA Sur'),
  ('belgrano', 'capital federal', 'CABA Norte')
AS t(partido, region, ciudad_nueva);

In [0]:
%sql

merge into bootcamp.gold.dim_zona_scd3 AS target
USING source_cambios as source
on target.partido=source.partido 
and target.region=source.region
WHEN matched 
AND target.ciudad!=source.ciudad_nueva
THEN update set
ciudad_anterior=target.ciudad,
ciudad=source.ciudad_nueva,
fecha_ultima_actualizacion=current_timestamp()
WHEN not matched 
THEN insert(partido,region,ciudad)
VALUES(partido,region,ciudad_nueva)

In [0]:
%sql

select * from bootcamp.gold.dim_zona_scd3 


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW source_cambios_2 AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA SurOeste'),
  ('belgrano', 'capital federal', 'CABA NorEste')
AS t(partido, region, ciudad_nueva);

In [0]:
%sql

merge into bootcamp.gold.dim_zona_scd3 AS target
USING source_cambios_2 as source
on target.partido=source.partido 
and target.region=source.region
WHEN matched 
AND target.ciudad!=source.ciudad_nueva
THEN update set
ciudad_anterior=target.ciudad,
ciudad=source.ciudad_nueva,
fecha_ultima_actualizacion=current_timestamp()
WHEN not matched 
THEN insert(partido,region,ciudad)
VALUES(partido,region,ciudad_nueva)

In [0]:
%sql

select * from bootcamp.gold.dim_zona_scd3
where ciudad_anterior is not null 

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona (
 zona_id BIGINT generated always as identity,
 partido STRING NOT NULL,
 region STRING NOT NULL,
 ciudad STRING,
 provincia STRING DEFAULT "buenos aires",
 Pais STRING DEFAULT "Argentina",
 _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
 PRIMARY KEY (zona_id)
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension de zonas';
    


In [0]:
%sql

insert into bootcamp.gold.dim_zona
(partido, region, ciudad)
select distinct partido, region, 
case when region = "capital federal" then "CABA"
else "GBA" END
from bootcamp.silver.propiedades
WHERE partido IS NOT NULL

In [0]:
%sql

SELECT COUNT(*) AS total_zonas FROM bootcamp.gold.dim_zona;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tipo_operacion (

    tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY,
    tipo_operacion STRING,
    moneda STRING,
    categoria string,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY (tipo_operacion_id)
) USING DELTA
COMMENT "Tabla Dimesion de tipo de operaciones"
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')


In [0]:
%sql

INSERT INTO bootcamp.gold.dim_tipo_operacion (tipo_operacion, moneda,categoria)
SELECT distinct tipo_operacion, moneda,
CASE WHEN tipo_operacion LIKE "%temporal%" then "temporal"
when tipo_operacion IN ('alquiler','venta') then "residencial" 
else "otro"
END as categoria
 FROM bootcamp.silver.propiedades
 where tipo_operacion is not null

In [0]:
%sql
-- dim_caracteristicas DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_caracteristicas (
  caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  estado             STRING NOT NULL,
  cochera            BOOLEAN,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING delta
COMMENT 'Dimension de caracteristicas de la propiedad (estado + cochera) - Star Schema'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- dim_caracteristicas ETL
INSERT INTO bootcamp.gold.dim_caracteristicas (estado, cochera)
SELECT DISTINCT
  COALESCE(estado, 'sin especificar') AS estado,
  COALESCE(cochera, false) AS cochera
FROM bootcamp.silver.propiedades;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tiempo (
  fecha_id BIGINT not null,
  fecha date,
  anio INT,
  mes INT,
  dia INT,
  dia_semana STRING,
  es_fin_semana BOOLEAN,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (fecha_id)
)
USING DELTA
COMMENT 'Dimension de tiempo -- fecha_id formato YYYYMMDD'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');
 

In [0]:
%sql

INSERT INTO bootcamp.gold.dim_tiempo (fecha_id, fecha, anio, mes, trimestre, dia_semana, es_fin_de_semana)

select DISTINCT
    CAST(DATE_FORMAT(CAST(_processing_timestamp as DATE), "yyyyMMdd") AS BIGINT) as fecha_id,
    CAST(_processing_timestamp AS DATE) AS fecha,
    YEAR(_processing_timestamp) AS anio,
    MONTH(_processing_timestamp) AS mes,
    QUARTER(_processing_timestamp) AS trimestre,
    DATE_FORMAT(_processing_timestamp, 'EEEE') AS dia_semana,
    CASE WHEN DAYOFWEEK(_processing_timestamp) IN (1, 7) THEN TRUE ELSE FALSE END AS es_fin_de_semana
FROM bootcamp.silver.propiedades
WHERE _processing_timestamp IS NOT NULL;

In [0]:
%sql

SELECT 'dim_zona' AS dim_nombre, COUNT(*) AS total_registros, COUNT(*) - COUNT(partido) AS nulls_bk
FROM bootcamp.gold.dim_zona
UNION ALL
SELECT 'dim_tipo_operacion', COUNT(*), COUNT(*) - COUNT(tipo_operacion)
FROM bootcamp.gold.dim_tipo_operacion
UNION ALL
SELECT 'dim_caracteristicas', COUNT(*), COUNT(*) - COUNT(estado)
FROM bootcamp.gold.dim_caracteristicas
UNION ALL
SELECT 'dim_orientacion', COUNT(*), COUNT(*) - COUNT(orientacion)
FROM bootcamp.gold.dim_orientacion
UNION ALL
SELECT 'dim_tiempo', COUNT(*), COUNT(*) - COUNT(fecha)
FROM bootcamp.gold.dim_tiempo;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bootcamp.gold.fact_propiedades (
  row_hash                    STRING NOT NULL,
  zona_id                     BIGINT,
  tipo_operacion_id           BIGINT,
  fecha_id                    BIGINT,
  caracteristicas_id          BIGINT,
  orientacion_id              BIGINT,
  precio                      DECIMAL(15,2),
  expensas                    DECIMAL(15,2),
  precio_por_m2               DECIMAL(15,2),
  metros_cuadrados_totales    DECIMAL(15,2),
  metros_cuadrados_cubiertos  DECIMAL(15,2),
  ambientes                   INT,
  url                         STRING,
  _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING delta
COMMENT 'Tabla de hechos de propiedades - Star Schema. PK = row_hash (MD5 de url + precio)'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
INSERT INTO bootcamp.gold.fact_propiedades (
  row_hash, zona_id, tipo_operacion_id, fecha_id,
  caracteristicas_id, orientacion_id,
  precio, expensas, precio_por_m2,
  metros_cuadrados_totales, metros_cuadrados_cubiertos,
  ambientes, url
)
SELECT DISTINCT
  MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING))) AS row_hash,
  dz.zona_id,
  dt.tipo_operacion_id,
  CAST(DATE_FORMAT(sp.fecha_publicacion, 'yyyyMMdd') AS BIGINT) AS fecha_id,
  dc.caracteristicas_id,
  do.orientacion_id,
  sp.precio,
  sp.expensas,
  sp.precio_por_m2,
  sp.metros_cuadrados_totales,
  sp.metros_cuadrados_cubiertos,
  sp.ambientes,
  sp.url
FROM bootcamp.silver.propiedades sp
LEFT JOIN bootcamp.gold.dim_zona dz
  ON sp.partido = dz.partido AND sp.region = dz.region
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt
  ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
LEFT JOIN bootcamp.gold.dim_caracteristicas dc
  ON COALESCE(sp.estado, 'sin especificar') = dc.estado
  AND COALESCE(sp.cochera, false) = dc.cochera
LEFT JOIN bootcamp.gold.dim_orientacion do
  ON COALESCE(sp.orientacion, 'sin especificar') = do.orientacion
LEFT JOIN bootcamp.gold.dim_tiempo dtm
  ON sp.fecha_publicacion = dtm.fecha;

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  COUNT(zona_id) AS con_zona_id,
  COUNT(*) - COUNT(zona_id) AS sin_zona_id,
  COUNT(tipo_operacion_id) AS con_tipo_op_id,
  COUNT(*) - COUNT(tipo_operacion_id) AS sin_tipo_op_id,
  COUNT(caracteristicas_id) AS con_caract_id,
  COUNT(*) - COUNT(caracteristicas_id) AS sin_caract_id,
  COUNT(orientacion_id) AS con_orient_id,
  COUNT(*) - COUNT(orientacion_id) AS sin_orient_id,
  COUNT(fecha_id) AS con_fecha_id,
  COUNT(*) - COUNT(fecha_id) AS sin_fecha_id
FROM bootcamp.gold.fact_propiedades;

In [0]:
%sql
-- Comparar LEFT JOIN vs INNER JOIN: cuantos registros se pierden?
SELECT
  (SELECT COUNT(*) FROM bootcamp.gold.fact_propiedades) AS con_left_join,
  (SELECT COUNT(*)
   FROM bootcamp.silver.propiedades sp
   INNER JOIN bootcamp.gold.dim_zona dz ON sp.partido = dz.partido AND sp.region = dz.region
   INNER JOIN bootcamp.gold.dim_tipo_operacion dt ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
   INNER JOIN bootcamp.gold.dim_caracteristicas dc ON COALESCE(sp.estado, 'sin especificar') = dc.estado AND COALESCE(sp.cochera, false) = dc.cochera
  ) AS con_inner_join;